# Parkinson's Spiral Drawing CNN

## Goal
Train a CNN to classify spiral drawings as healthy or Parkinson's-related using transfer learning.

## Evaluation Strategy
Model performance is reported using 5-fold StratifiedKFold cross-validation. Each image is evaluated only in the fold where it was held out.

## Leakage Prevention
Augmentation is applied only to training folds. Validation folds use deterministic preprocessing only.

## Final Deployment Model
After evaluation, the final model is retrained on all available images for use in the Streamlit app. No performance metrics are reported from this final model.

In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
import json
import pandas as pd
import copy
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve
import random


In [13]:
base = "../data/archive (1)/spiral"

train_healthy = [os.path.join(base, "training/healthy", f) for f in os.listdir(os.path.join(base, "training/healthy")) if f.endswith('.png')]
train_parkinson = [os.path.join(base, "training/parkinson", f) for f in os.listdir(os.path.join(base, "training/parkinson")) if f.endswith('.png')]
test_healthy = [os.path.join(base, "testing/healthy", f) for f in os.listdir(os.path.join(base, "testing/healthy")) if f.endswith('.png')]
test_parkinson = [os.path.join(base, "testing/parkinson", f) for f in os.listdir(os.path.join(base, "testing/parkinson")) if f.endswith('.png')]

train_paths = train_healthy + train_parkinson
train_labels = [0]*len(train_healthy) + [1]*len(train_parkinson)
test_paths = test_healthy + test_parkinson
test_labels = [0]*len(test_healthy) + [1]*len(test_parkinson)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

print(f"Training: {len(train_paths)} images ({len(train_healthy)} healthy, {len(train_parkinson)} parkinson)")
print(f"Testing: {len(test_paths)} images ({len(test_healthy)} healthy, {len(test_parkinson)} parkinson)")

Training: 72 images (36 healthy, 36 parkinson)
Testing: 30 images (15 healthy, 15 parkinson)


In [14]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# Strong augmentation for training (small dataset needs lots of variation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# No augmentation for testing
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class SpiralDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

Using device: mps


In [15]:
all_paths = train_paths + test_paths
all_labels = train_labels + test_labels
criterion = nn.CrossEntropyLoss()
print(f"Total images: {len(all_paths)} ({sum(1 for l in all_labels if l == 0)} healthy, {sum(1 for l in all_labels if l == 1)} parkinson)")

Total images: 102 (51 healthy, 51 parkinson)


In [16]:
skf_roc = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_oof_probs = np.zeros(len(all_paths))
all_oof_labels = np.array(all_labels)

fold_results = []
fold_precisions = []
fold_recalls = []
fold_f1s = []

for fold_idx, (train_idx, val_idx) in enumerate(skf_roc.split(all_paths, all_labels)):
    print(f"\n========== Fold {fold_idx + 1}/5 ==========")
    
    fold_train_paths = [all_paths[i] for i in train_idx]
    fold_train_labels = [all_labels[i] for i in train_idx]
    fold_test_paths = [all_paths[i] for i in val_idx]
    fold_test_labels = [all_labels[i] for i in val_idx]
    
    fold_train_dataset = SpiralDataset(fold_train_paths, fold_train_labels, transform=train_transform)
    fold_test_dataset = SpiralDataset(fold_test_paths, fold_test_labels, transform=test_transform)
    fold_train_loader = DataLoader(fold_train_dataset, batch_size=8, shuffle=True)
    fold_test_loader = DataLoader(fold_test_dataset, batch_size=8, shuffle=False)
    
    fold_model = models.mobilenet_v2(weights='IMAGENET1K_V1')
    for param in fold_model.parameters():
        param.requires_grad = False
    fold_model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(fold_model.classifier[1].in_features, 2)
    )
    fold_model = fold_model.to(device)
    
    # Phase 1 — frozen training (15 fixed epochs)
    optimizer_f = optim.Adam(fold_model.classifier.parameters(), lr=0.001, weight_decay=0.01)
    for epoch in range(15):
        fold_model.train()
        for images, labels in fold_train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer_f.zero_grad()
            outputs = fold_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer_f.step()
    
    # Phase 2 — fine-tuning (20 fixed epochs, no validation-based selection)
    for param in list(fold_model.parameters())[-30:]:
        param.requires_grad = True
    optimizer_f = optim.Adam([p for p in fold_model.parameters() if p.requires_grad], lr=0.0001, weight_decay=0.01)
    
    for epoch in range(20):
        fold_model.train()
        for images, labels in fold_train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer_f.zero_grad()
            outputs = fold_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer_f.step()
    
    fold_model.eval()
    fold_probs = []
    fold_preds = []
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in fold_test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = fold_model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = (probs >= 0.5).long()
            
            fold_probs.extend(probs.cpu().numpy())
            fold_preds.extend(preds.cpu().numpy())
            
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    
    fold_acc = 100 * correct / total
    
    # Save probabilities at original positions
    for i, idx in enumerate(val_idx):
        all_oof_probs[idx] = fold_probs[i]
    
    # Compute additional metrics for this fold
    fold_precision = precision_score(fold_test_labels, fold_preds, zero_division=0)
    fold_recall = recall_score(fold_test_labels, fold_preds, zero_division=0)
    fold_f1 = f1_score(fold_test_labels, fold_preds, zero_division=0)
    
    fold_precisions.append(fold_precision)
    fold_recalls.append(fold_recall)
    fold_f1s.append(fold_f1)
    
    print(f"Fold {fold_idx + 1} done. Acc: {fold_acc:.2f}%, Precision: {fold_precision:.3f}, Recall: {fold_recall:.3f}, F1: {fold_f1:.3f}")
    fold_results.append(fold_acc)

# Compute ROC from cross-validated predictions
fpr_drawing, tpr_drawing, _ = roc_curve(all_oof_labels, all_oof_probs)
auc_oof = roc_auc_score(all_oof_labels, all_oof_probs)
print(f"\nOut-of-fold AUC: {auc_oof:.3f}")
print(f"Mean fold accuracy: {np.mean(fold_results):.2f}%")
print(f"Std fold accuracy: {np.std(fold_results):.2f}%")

# Save ROC data
roc_drawing = pd.DataFrame({'FPR': fpr_drawing, 'TPR': tpr_drawing})
roc_drawing.to_csv('../models/roc_drawing.csv', index=False)


========== Fold 1/5 ==========
Fold 1 done. Acc: 85.71%, Precision: 1.000, Recall: 0.700, F1: 0.824

========== Fold 2/5 ==========
Fold 2 done. Acc: 90.48%, Precision: 0.846, Recall: 1.000, F1: 0.917

========== Fold 3/5 ==========
Fold 3 done. Acc: 85.00%, Precision: 1.000, Recall: 0.700, F1: 0.824

========== Fold 4/5 ==========
Fold 4 done. Acc: 80.00%, Precision: 0.750, Recall: 0.900, F1: 0.818

========== Fold 5/5 ==========
Fold 5 done. Acc: 70.00%, Precision: 0.833, Recall: 0.500, F1: 0.625

Out-of-fold AUC: 0.911
Mean fold accuracy: 82.24%
Std fold accuracy: 6.96%


In [17]:
all_oof_preds = (all_oof_probs >= 0.5).astype(int)
cm_clean = confusion_matrix(all_oof_labels, all_oof_preds)
cm_clean_df = pd.DataFrame(cm_clean,
                            index=['Actual Healthy', 'Actual PD'],
                            columns=['Predicted Healthy', 'Predicted PD'])
cm_clean_df.to_csv('../models/confusion_matrix_drawing.csv')
print(cm_clean_df)

                Predicted Healthy  Predicted PD
Actual Healthy                 45             6
Actual PD                      12            39


In [18]:
cv_results = {
    'method': '5-fold StratifiedKFold cross-validation',
    'dataset': 'Original Parkinson\'s spiral drawings (102 patients, no augmentation leakage)',
    'leakage_prevention': 'Augmentation applied only inside training folds; validation folds used deterministic transforms only.',
    'fold_accuracies': [round(a, 2) for a in fold_results],
    'mean_accuracy': round(np.mean(fold_results), 2),
    'std_accuracy': round(np.std(fold_results), 2),
    'mean_precision': round(np.mean(fold_precisions), 3),
    'mean_recall': round(np.mean(fold_recalls), 3),
    'mean_f1': round(np.mean(fold_f1s), 3),
    'roc_auc': round(auc_oof, 3)
}

with open('../models/cv_results.json', 'w') as f:
    json.dump(cv_results, f, indent=2)

print(json.dumps(cv_results, indent=2))

{
  "method": "5-fold StratifiedKFold cross-validation",
  "dataset": "Original Parkinson's spiral drawings (102 patients, no augmentation leakage)",
  "leakage_prevention": "Augmentation applied only inside training folds; validation folds used deterministic transforms only.",
  "fold_accuracies": [
    85.71,
    90.48,
    85.0,
    80.0,
    70.0
  ],
  "mean_accuracy": 82.24,
  "std_accuracy": 6.96,
  "mean_precision": 0.886,
  "mean_recall": 0.76,
  "mean_f1": 0.801,
  "roc_auc": 0.911
}


In [19]:
# Final model trained on all spiral data
final_model = models.mobilenet_v2(weights='IMAGENET1K_V1')
for param in final_model.parameters():
    param.requires_grad = False
final_model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(final_model.classifier[1].in_features, 2)
)
final_model = final_model.to(device)

# Use ALL data for final model
all_dataset = SpiralDataset(all_paths, all_labels, transform=train_transform)
all_loader = DataLoader(all_dataset, batch_size=8, shuffle=True)

# Phase 1 — frozen
optimizer_final = optim.Adam(final_model.classifier.parameters(), lr=0.001, weight_decay=0.01)
print("Phase 1: Frozen training")
for epoch in range(15):
    final_model.train()
    for images, labels in all_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer_final.zero_grad()
        outputs = final_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_final.step()
    print(f"Epoch {epoch+1}/15 done")

# Phase 2 — fine-tune
for param in list(final_model.parameters())[-30:]:
    param.requires_grad = True
optimizer_final = optim.Adam([p for p in final_model.parameters() if p.requires_grad], lr=0.0001, weight_decay=0.01)

print("\nPhase 2: Fine-tuning")
for epoch in range(20):
    final_model.train()
    for images, labels in all_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer_final.zero_grad()
        outputs = final_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_final.step()
    print(f"Epoch {epoch+1}/20 done")

# Save
torch.save(final_model.state_dict(), '../models/drawing_cnn_final.pth')

Phase 1: Frozen training
Epoch 1/15 done
Epoch 2/15 done
Epoch 3/15 done
Epoch 4/15 done
Epoch 5/15 done
Epoch 6/15 done
Epoch 7/15 done
Epoch 8/15 done
Epoch 9/15 done
Epoch 10/15 done
Epoch 11/15 done
Epoch 12/15 done
Epoch 13/15 done
Epoch 14/15 done
Epoch 15/15 done

Phase 2: Fine-tuning
Epoch 1/20 done
Epoch 2/20 done
Epoch 3/20 done
Epoch 4/20 done
Epoch 5/20 done
Epoch 6/20 done
Epoch 7/20 done
Epoch 8/20 done
Epoch 9/20 done
Epoch 10/20 done
Epoch 11/20 done
Epoch 12/20 done
Epoch 13/20 done
Epoch 14/20 done
Epoch 15/20 done
Epoch 16/20 done
Epoch 17/20 done
Epoch 18/20 done
Epoch 19/20 done
Epoch 20/20 done
